# IFRS S1/S2 Agentic Report Generation — Clean LangGraph Rewrite

This notebook is a clean rewrite of the report-generation pipeline. It keeps the core output contract from the previous working version while removing historical override chains.

Architecture:

1. Deterministic preparation: load inputs, map evidence, classify coverage, build disclosure plans.
2. LangGraph orchestration: write section, build claims, run gates, judge, approve, revise, or route to review.
3. Deterministic finalisation: quality refinement, final QA, editorial polish, final Markdown assembly, audit summary.

The section-generation loop is the only part converted to LangGraph. Deterministic preparation and final report controls remain normal Python because they are batch validation/reporting phases rather than agentic loops.

In [ ]:
# ============================================================
# CELL 1 — CONFIGURATION, PATHS, AND OUTPUT CONTRACT
# ============================================================

from __future__ import annotations

import copy
import json
import math
import os
import re
import time
import urllib.error
import urllib.request
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from decimal import Decimal, InvalidOperation
from pathlib import Path
from typing import Any, Dict, Iterable, List, Literal, Optional, Sequence, Tuple, TypedDict

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from dotenv import find_dotenv, load_dotenv
    try:
        load_dotenv(find_dotenv(usecwd=True), override=True)
    except Exception:
        load_dotenv(Path.cwd() / ".env", override=True)
except ImportError:
    pass

CURRENT_DIR = Path.cwd().resolve()
NOTEBOOK_DIR = CURRENT_DIR if CURRENT_DIR.name == "notebooks" else (CURRENT_DIR / "notebooks" if (CURRENT_DIR / "notebooks").exists() else CURRENT_DIR)
NOTEBOOK_DIR = NOTEBOOK_DIR.resolve()
GEN_DATA_DIR = Path(os.getenv("GEN_DATA_DIR", NOTEBOOK_DIR / "gen_data")).resolve()

PAYLOAD_DIR = Path(os.getenv("PAYLOAD_DIR", GEN_DATA_DIR / "payloads")).resolve()
REQUIREMENTS_DIR = Path(os.getenv("IFRS_REQUIREMENTS_DIR", GEN_DATA_DIR / "IFRS" / "ifrs_requirements_kb_outputs_final" / "section_by_section_requirements" / "json")).resolve()
STYLE_SYSTEM_DIR = Path(os.getenv("STYLE_SYSTEM_DIR", GEN_DATA_DIR / "style" / "style_system")).resolve()
OUTPUT_DIR = Path(os.getenv("GENERATION_OUTPUT_DIR", GEN_DATA_DIR / "generated_reports" / "agentic_ifrs_report")).resolve()

SECTIONS: List[str] = [
    "General Requirements",
    "Governance",
    "Strategy",
    "Risk Management",
    "Metrics and Targets",
]

SECTION_SLUGS: Dict[str, str] = {
    "General Requirements": "general_requirements",
    "Governance": "governance",
    "Strategy": "strategy",
    "Risk Management": "risk_management",
    "Metrics and Targets": "metrics_and_targets",
}

PAYLOAD_FILE_HINTS: Dict[str, List[str]] = {
    "General Requirements": ["payload_BANK01_general_requirements.json", "general_requirements"],
    "Governance": ["payload_BANK01_governance.json", "governance"],
    "Strategy": ["payload_BANK01_strategy.json", "strategy"],
    "Risk Management": ["payload_BANK01_risk_management.json", "risk_management"],
    "Metrics and Targets": ["payload_BANK01_metrics_targets.json", "metrics_targets", "metrics_and_targets"],
}

@dataclass(frozen=True)
class PipelineConfig:
    entity_name: str = "Eurolux Universal Bank AG"
    bank_id: str = "BANK01"
    reporting_year: int = 2024
    currency: str = "EUR"
    max_revision_loops: int = int(os.getenv("MAX_REVISION_LOOPS", "2"))
    target_section_score: float = float(os.getenv("TARGET_SECTION_SCORE", "85"))
    min_supported_judge_floor: float = float(os.getenv("IFRS_SUPPORTED_SCOPE_JUDGE_FLOOR", "5.5"))
    factlock_relative_tolerance: float = float(os.getenv("IFRS_FACTLOCK_REL_TOL", "0.002"))
    factlock_absolute_tolerance: float = float(os.getenv("IFRS_FACTLOCK_ABS_TOL", "0.2"))
    offline_mode: bool = os.getenv("IFRS_OFFLINE_MODE", "0").strip().lower() in {"1", "true", "yes"}
    strict_final_quality: bool = os.getenv("IFRS_STRICT_FINAL_QUALITY", "1").strip().lower() not in {"0", "false", "no"}

CONFIG = PipelineConfig()

DIRS: Dict[str, Path] = {
    "evidence_maps": OUTPUT_DIR / "01_evidence_maps",
    "coverage": OUTPUT_DIR / "02_coverage",
    "missing_requirements": OUTPUT_DIR / "03_missing_requirements",
    "disclosure_plans": OUTPUT_DIR / "04_disclosure_plans",
    "draft_sections": OUTPUT_DIR / "05_draft_sections",
    "claims_registers": OUTPUT_DIR / "06_claims_registers",
    "deterministic_gates": OUTPUT_DIR / "07_deterministic_gates",
    "judge_results": OUTPUT_DIR / "08_judge_results",
    "revised_sections": OUTPUT_DIR / "09_revised_sections",
    "approved_sections": OUTPUT_DIR / "10_approved_sections",
    "connectivity": OUTPUT_DIR / "11_connectivity",
    "pdf_handoff": OUTPUT_DIR / "12_pdf_handoff",
    "final_quality": OUTPUT_DIR / "13_final_quality",
    "quality_refinement": OUTPUT_DIR / "14_quality_refinement",
    "final_editorial": OUTPUT_DIR / "15_final_editorial",
    "audit_logs": OUTPUT_DIR / "audit_logs",
}

for directory in DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

print("Notebook directory:", NOTEBOOK_DIR)
print("Output directory:", OUTPUT_DIR)
print("Sections:", ", ".join(SECTIONS))


In [ ]:
# ============================================================
# CELL 2 — GENERAL UTILITIES
# ============================================================

def section_slug(section_name: str) -> str:
    if section_name not in SECTION_SLUGS:
        raise ValueError(f"Unknown section: {section_name}")
    return SECTION_SLUGS[section_name]


def read_json(path: Path, default: Any = None) -> Any:
    path = Path(path)
    if not path.exists():
        if default is not None:
            return copy.deepcopy(default)
        raise FileNotFoundError(path)
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def write_json(obj: Any, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def read_text(path: Path, default: str = "") -> str:
    path = Path(path)
    if not path.exists():
        return default
    return path.read_text(encoding="utf-8")


def write_text(text: str, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text or "", encoding="utf-8")


def value_preview(value: Any, limit: int = 220) -> str:
    if isinstance(value, (dict, list)):
        text = json.dumps(value, ensure_ascii=False)
    else:
        text = str(value)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:limit] + ("…" if len(text) > limit else "")


def is_empty_value(value: Any) -> bool:
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    if isinstance(value, str) and not value.strip():
        return True
    if isinstance(value, (list, tuple, set, dict)) and len(value) == 0:
        return True
    return False


def flatten_json(obj: Any, prefix: str = "") -> Dict[str, Any]:
    flat: Dict[str, Any] = {}
    if isinstance(obj, dict):
        for key, value in obj.items():
            path = f"{prefix}.{key}" if prefix else str(key)
            flat.update(flatten_json(value, path))
    elif isinstance(obj, list):
        for index, value in enumerate(obj):
            path = f"{prefix}[{index}]"
            flat.update(flatten_json(value, path))
    else:
        flat[prefix] = obj
    return flat


def normalize_text(text: Any) -> str:
    return re.sub(r"\s+", " ", str(text or "").lower()).strip()


STOPWORDS = {
    "the", "and", "for", "with", "that", "this", "from", "into", "are", "its", "has", "have", "shall", "must", "should",
    "entity", "entities", "information", "disclose", "disclosure", "related", "including", "about", "which", "their", "within",
    "report", "reporting", "financial", "sustainability", "climate", "risk", "risks", "opportunity", "opportunities",
}


def tokens(text: Any) -> List[str]:
    words = re.findall(r"[a-zA-Z][a-zA-Z0-9_\-]{2,}", normalize_text(text))
    return [w.replace("_", "-") for w in words if w not in STOPWORDS]


def token_set(text: Any) -> set:
    return set(tokens(text))


def now_stamp() -> str:
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())


In [ ]:
# ============================================================
# CELL 3 — AZURE OPENAI CLIENT AND ROBUST JSON PARSING
# ============================================================

MODEL_CONFIG = {
    "section_writer": "strong",
    "claims_register_builder": "strong",
    "ifrs_coverage_judge": "strong",
    "evidence_judge": "strong",
    "style_judge": "fast",
    "minimal_reviser": "strong",
    "final_quality_reconciler": "strong",
    "final_editorial_polisher": "strong",
    "whole_report_connectivity_judge": "strong",
}


def _deployment_url_for(agent_name: str) -> str:
    tier = MODEL_CONFIG.get(agent_name, "strong")
    env_key = "AZURE_OPENAI_FAST_DEPLOYMENT_URL" if tier == "fast" else "AZURE_OPENAI_GPT52_DEPLOYMENT_URL"
    return os.getenv(env_key, "").strip()


def validate_llm_config(required: bool = True) -> Dict[str, Any]:
    status = {
        "api_key_present": bool(os.getenv("AZURE_OPENAI_API_KEY", "").strip()),
        "strong_url_present": bool(os.getenv("AZURE_OPENAI_GPT52_DEPLOYMENT_URL", "").strip()),
        "fast_url_present": bool(os.getenv("AZURE_OPENAI_FAST_DEPLOYMENT_URL", "").strip()),
        "offline_mode": CONFIG.offline_mode,
    }
    if required and not CONFIG.offline_mode and not (status["api_key_present"] and status["strong_url_present"]):
        raise RuntimeError(
            "Azure OpenAI configuration is incomplete. Set AZURE_OPENAI_API_KEY and "
            "AZURE_OPENAI_GPT52_DEPLOYMENT_URL, or set IFRS_OFFLINE_MODE=1 for deterministic smoke tests."
        )
    return status


def azure_chat(
    agent_name: str,
    messages: List[Dict[str, str]],
    *,
    temperature: float = 0.1,
    max_tokens: int = 4000,
    response_format: Optional[Dict[str, str]] = None,
) -> str:
    if CONFIG.offline_mode:
        raise RuntimeError("LLM call requested while IFRS_OFFLINE_MODE=1")

    api_key = os.getenv("AZURE_OPENAI_API_KEY", "").strip()
    url = _deployment_url_for(agent_name)
    if not api_key or not url:
        raise RuntimeError(f"Missing Azure OpenAI configuration for agent '{agent_name}'.")

    payload: Dict[str, Any] = {
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    if response_format:
        payload["response_format"] = response_format

    request = urllib.request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json", "api-key": api_key},
        method="POST",
    )
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            data = json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Azure OpenAI HTTP {e.code}: {body[:1000]}") from e

    try:
        return data["choices"][0]["message"]["content"]
    except Exception as e:
        raise RuntimeError(f"Unexpected Azure OpenAI response structure: {data}") from e


def strip_markdown_json_fence(text: str) -> str:
    text = (text or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```$", "", text)
    return text.strip()


def extract_json_object(text: str) -> str:
    text = strip_markdown_json_fence(text)
    if text.startswith("{") and text.endswith("}"):
        return text
    start = text.find("{")
    if start < 0:
        raise ValueError("No JSON object found in LLM response.")
    depth = 0
    in_string = False
    escape = False
    for pos in range(start, len(text)):
        char = text[pos]
        if in_string:
            if escape:
                escape = False
            elif char == "\\":
                escape = True
            elif char == '"':
                in_string = False
        else:
            if char == '"':
                in_string = True
            elif char == "{":
                depth += 1
            elif char == "}":
                depth -= 1
                if depth == 0:
                    return text[start : pos + 1]
    raise ValueError("Unbalanced JSON object in LLM response.")


def parse_llm_json(text: str, *, audit_name: str = "llm_json") -> Dict[str, Any]:
    try:
        return json.loads(extract_json_object(text))
    except Exception as e:
        audit_dir = DIRS["audit_logs"] / "llm_json_outputs"
        audit_dir.mkdir(parents=True, exist_ok=True)
        raw_path = audit_dir / f"{audit_name}_{int(time.time())}.txt"
        write_text(text or "", raw_path)
        raise ValueError(f"Could not parse JSON. Raw output saved to {raw_path}") from e


def azure_chat_json(
    agent_name: str,
    messages: List[Dict[str, str]],
    *,
    temperature: float = 0.05,
    max_tokens: int = 3000,
    audit_name: str = "llm_json",
) -> Dict[str, Any]:
    text = azure_chat(
        agent_name,
        messages,
        temperature=temperature,
        max_tokens=max_tokens,
        response_format={"type": "json_object"},
    )
    return parse_llm_json(text, audit_name=audit_name)

print(validate_llm_config(required=False))


In [ ]:
# ============================================================
# CELL 4 — INPUT LOADERS
# ============================================================

def input_search_roots() -> List[Path]:
    roots = [NOTEBOOK_DIR, GEN_DATA_DIR, PAYLOAD_DIR, REQUIREMENTS_DIR, STYLE_SYSTEM_DIR]
    sandbox_root = Path("/mnt/data")
    if sandbox_root.exists():
        roots.append(sandbox_root)
    unique: List[Path] = []
    seen = set()
    for root in roots:
        root = Path(root).resolve()
        if root not in seen:
            unique.append(root)
            seen.add(root)
    return unique


def find_first_file(candidates: Sequence[str], roots: Optional[Sequence[Path]] = None) -> Optional[Path]:
    roots = list(roots or input_search_roots())
    exact_candidates = [c for c in candidates if c.endswith(".json") or c.endswith(".md") or c.endswith(".txt")]
    fuzzy_candidates = [c.lower() for c in candidates]

    for root in roots:
        for candidate in exact_candidates:
            path = root / candidate
            if path.exists():
                return path

    for root in roots:
        if not root.exists():
            continue
        for path in root.rglob("*"):
            if not path.is_file():
                continue
            name = path.name.lower()
            if any(c in name for c in fuzzy_candidates):
                return path
    return None


def rows_from_requirements_object(obj: Any) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    if isinstance(obj, list):
        for item in obj:
            rows.extend(rows_from_requirements_object(item))
    elif isinstance(obj, dict):
        if "requirement_text" in obj or "requirement_id" in obj:
            rows.append(obj)
        for key, value in obj.items():
            if key in {"requirements", "rows", "items"} or isinstance(value, (dict, list)):
                rows.extend(rows_from_requirements_object(value))
    return rows


def normalize_requirement(row: Dict[str, Any], section_name: str) -> Dict[str, Any]:
    requirement_id = str(row.get("requirement_id") or row.get("id") or row.get("paragraph_id") or f"{section_slug(section_name)}_{abs(hash(json.dumps(row, sort_keys=True, default=str))) % 10**8}")
    requirement_text = str(row.get("requirement_text") or row.get("text") or row.get("description") or "").strip()
    return {
        "requirement_id": requirement_id,
        "standard": row.get("standard"),
        "paragraph_id": row.get("paragraph_id"),
        "report_section": row.get("report_section") or section_name,
        "official_section_heading": row.get("official_section_heading"),
        "nearest_pdf_heading": row.get("nearest_pdf_heading"),
        "requirement_text": requirement_text,
        "raw": row,
    }


def load_requirements_for_section(section_name: str) -> List[Dict[str, Any]]:
    slug = section_slug(section_name)
    path = find_first_file([f"{slug}_requirements.json", slug, "requirements"])
    if path is None:
        raise FileNotFoundError(f"No requirements file found for {section_name}")
    obj = read_json(path)
    rows = rows_from_requirements_object(obj)
    normalized = [normalize_requirement(row, section_name) for row in rows]
    filtered = [r for r in normalized if r["requirement_text"]]
    return filtered


def load_payload_for_section(section_name: str) -> Dict[str, Any]:
    hints = PAYLOAD_FILE_HINTS[section_name]
    path = find_first_file(hints + [section_slug(section_name), "payload"])
    if path is None:
        raise FileNotFoundError(f"No payload file found for {section_name}")
    return read_json(path)


def load_style_for_section(section_name: str) -> Dict[str, Any]:
    slug = section_slug(section_name)
    style: Dict[str, Any] = {}
    possible_names = [f"{slug}_style.json", f"{slug}_blueprint.json", slug]
    for path in input_search_roots():
        if not path.exists():
            continue
        for file_path in path.rglob("*.json"):
            name = file_path.name.lower()
            if any(candidate.lower() in name for candidate in possible_names):
                try:
                    style[file_path.stem] = read_json(file_path)
                except Exception:
                    pass
    return style


def load_all_inputs() -> Tuple[Dict[str, List[Dict[str, Any]]], Dict[str, Dict[str, Any]], Dict[str, Dict[str, Any]]]:
    requirements_by_section = {section: load_requirements_for_section(section) for section in SECTIONS}
    payloads_by_section = {section: load_payload_for_section(section) for section in SECTIONS}
    style_by_section = {section: load_style_for_section(section) for section in SECTIONS}

    summary = {
        section: {
            "requirements": len(requirements_by_section[section]),
            "payload_top_level_keys": list(payloads_by_section[section].keys())[:20],
            "style_artifacts": list(style_by_section[section].keys())[:20],
        }
        for section in SECTIONS
    }
    write_json(summary, OUTPUT_DIR / "input_loading_summary.json")
    return requirements_by_section, payloads_by_section, style_by_section

requirements_by_section, payloads_by_section, style_by_section = load_all_inputs()
print(json.dumps({s: len(requirements_by_section[s]) for s in SECTIONS}, indent=2))


In [ ]:
# ============================================================
# CELL 5 — STRICT EVIDENCE MAPPING AND COVERAGE CLASSIFICATION
# ============================================================

MISSING_LIKE_STRINGS = {
    "", "none", "null", "nan", "n/a", "na", "not available", "unavailable", "not applicable", "missing", "unknown",
}

AUDIT_ONLY_PATH_FRAGMENTS = {
    "data_gaps", "missing", "not_available", "unavailable", "limitation", "limitations", "instruction", "reason",
}

GENERIC_CONTEXT_LEAVES = {
    "reporting_year", "comparative_years", "currency", "bank_id", "entity_name", "fiscal_year_end",
}

SECTION_ROUTE_TERMS = {
    "General Requirements": {"basis", "preparation", "currency", "comparative", "boundary", "methodology", "assurance", "connected"},
    "Governance": {"board", "committee", "governance", "oversight", "remuneration", "skills", "management", "agenda"},
    "Strategy": {"strategy", "scenario", "resilience", "business", "model", "value", "chain", "opportunity", "financial", "planning"},
    "Risk Management": {"risk", "register", "identify", "assess", "prioritise", "monitor", "erm", "likelihood", "severity"},
    "Metrics and Targets": {"emissions", "scope", "target", "metric", "carbon", "financed", "intensity", "capex", "credit"},
}


def is_missing_like_value(value: Any) -> bool:
    if is_empty_value(value):
        return True
    if isinstance(value, str) and normalize_text(value) in MISSING_LIKE_STRINGS:
        return True
    return False


def is_audit_only_evidence_path(path: str) -> bool:
    lower = path.lower()
    return any(fragment in lower for fragment in AUDIT_ONLY_PATH_FRAGMENTS)


def writer_evidence_path_allowed(path: str, value: Any) -> bool:
    return not is_audit_only_evidence_path(path) and not is_missing_like_value(value)


def requirement_keywords(requirement: Dict[str, Any]) -> set:
    text = " ".join(
        str(requirement.get(key) or "")
        for key in ["requirement_id", "requirement_text", "official_section_heading", "nearest_pdf_heading", "standard"]
    )
    return token_set(text)


def evidence_keywords(path: str, value: Any) -> set:
    return token_set(path.replace(".", " ").replace("_", " ") + " " + value_preview(value, 200))


def evidence_strength_from_score(score: float) -> str:
    if score >= 6.0:
        return "strong"
    if score >= 3.0:
        return "medium"
    if score > 0:
        return "weak"
    return "none"


def score_evidence_candidate(section_name: str, requirement: Dict[str, Any], path: str, value: Any) -> float:
    if not writer_evidence_path_allowed(path, value):
        return 0.0
    req_tokens = requirement_keywords(requirement)
    ev_tokens = evidence_keywords(path, value)
    overlap = len(req_tokens & ev_tokens)
    score = float(overlap)
    path_lower = path.lower()
    requirement_text = normalize_text(requirement.get("requirement_text"))

    for term in SECTION_ROUTE_TERMS.get(section_name, set()):
        if term in path_lower or term in requirement_text:
            score += 0.4

    phrase_boosts = {
        "board": ["board", "committee", "agenda", "remuneration"],
        "scenario": ["scenario", "ngfs", "resilience", "transition"],
        "greenhouse": ["emissions", "scope", "ghg", "co2"],
        "financed": ["financed", "pcaf", "loans", "investment"],
        "risk management": ["register", "likelihood", "severity", "monitoring"],
        "target": ["target", "progress", "baseline", "milestone"],
        "currency": ["currency", "eur", "presentation"],
        "comparative": ["comparative", "2022", "2023"],
    }
    for phrase, hints in phrase_boosts.items():
        if phrase in requirement_text and any(h in path_lower for h in hints):
            score += 3.0

    leaf = re.split(r"\.|\[", path)[-1].replace("]", "")
    if leaf in GENERIC_CONTEXT_LEAVES:
        score += 1.0 if any(t in requirement_text for t in ["currency", "reporting", "comparative", "period", "boundary"]) else -2.0

    if isinstance(value, (int, float)):
        score += 0.5
    elif isinstance(value, str) and len(value) > 30:
        score += 0.5
    return max(score, 0.0)


def build_evidence_map_for_section(section_name: str) -> Dict[str, Any]:
    requirements = requirements_by_section[section_name]
    payload = payloads_by_section[section_name]
    flat_payload = flatten_json(payload)
    rows: List[Dict[str, Any]] = []

    for requirement in requirements:
        candidates: List[Dict[str, Any]] = []
        for path, value in flat_payload.items():
            score = score_evidence_candidate(section_name, requirement, path, value)
            if score <= 0:
                continue
            candidates.append({
                "path": path,
                "value_preview": value_preview(value),
                "raw_value": value,
                "score": round(score, 3),
                "strength": evidence_strength_from_score(score),
                "writer_safe": writer_evidence_path_allowed(path, value),
            })
        candidates.sort(key=lambda x: x["score"], reverse=True)
        rows.append({
            "requirement_id": requirement["requirement_id"],
            "requirement_text": requirement["requirement_text"],
            "standard": requirement.get("standard"),
            "paragraph_id": requirement.get("paragraph_id"),
            "evidence_candidates": candidates[:12],
            "best_strength": candidates[0]["strength"] if candidates else "none",
        })

    result = {
        "section": section_name,
        "generated_at": now_stamp(),
        "requirements_count": len(requirements),
        "requirements": rows,
    }
    slug = section_slug(section_name)
    write_json(result, DIRS["evidence_maps"] / f"evidence_map_{slug}.json")
    summary = Counter(row["best_strength"] for row in rows)
    write_json({"section": section_name, "best_strength_counts": dict(summary)}, DIRS["evidence_maps"] / f"evidence_map_summary_{slug}.json")
    return result


def classify_requirement_coverage(evidence_row: Dict[str, Any]) -> str:
    strengths = [c.get("strength") for c in evidence_row.get("evidence_candidates", []) if c.get("writer_safe")]
    if "strong" in strengths:
        return "covered"
    if "medium" in strengths:
        return "partially_covered"
    return "not_available_in_payload"


def build_coverage_and_missing_for_section(section_name: str, evidence_map: Dict[str, Any]) -> Dict[str, Any]:
    coverage_rows: List[Dict[str, Any]] = []
    missing_rows: List[Dict[str, Any]] = []
    for evidence_row in evidence_map["requirements"]:
        status = classify_requirement_coverage(evidence_row)
        item = {
            "requirement_id": evidence_row["requirement_id"],
            "requirement_text": evidence_row["requirement_text"],
            "coverage_status": status,
            "best_strength": evidence_row["best_strength"],
            "evidence_paths": [c["path"] for c in evidence_row.get("evidence_candidates", []) if c.get("writer_safe")][:5],
        }
        coverage_rows.append(item)
        if status != "covered":
            missing_rows.append({
                **item,
                "audit_only": True,
                "report_instruction": "Do not mention this gap in report prose. Use supported evidence only.",
            })

    covered = sum(1 for r in coverage_rows if r["coverage_status"] == "covered")
    partial = sum(1 for r in coverage_rows if r["coverage_status"] == "partially_covered")
    total = len(coverage_rows) or 1
    result = {
        "section": section_name,
        "generated_at": now_stamp(),
        "coverage_score": round((covered + 0.5 * partial) / total * 100, 2),
        "covered_count": covered,
        "partial_count": partial,
        "not_available_count": len(coverage_rows) - covered - partial,
        "requirements": coverage_rows,
    }
    slug = section_slug(section_name)
    write_json(result, DIRS["coverage"] / f"coverage_matrix_{slug}.json")
    write_json({"section": section_name, "requirements": missing_rows}, DIRS["missing_requirements"] / f"missing_requirements_{slug}.json")
    return result


def prepare_evidence_and_coverage() -> Dict[str, Dict[str, Any]]:
    results: Dict[str, Dict[str, Any]] = {}
    all_missing: List[Dict[str, Any]] = []
    for section in SECTIONS:
        evidence_map = build_evidence_map_for_section(section)
        coverage = build_coverage_and_missing_for_section(section, evidence_map)
        results[section] = {"evidence_map": evidence_map, "coverage": coverage}
        all_missing.extend(read_json(DIRS["missing_requirements"] / f"missing_requirements_{section_slug(section)}.json")["requirements"])
    write_json({"generated_at": now_stamp(), "requirements": all_missing}, DIRS["missing_requirements"] / "missing_requirements_all_sections.json")
    return results

evidence_and_coverage = prepare_evidence_and_coverage()
print({section: evidence_and_coverage[section]["coverage"]["coverage_score"] for section in SECTIONS})


In [ ]:
# ============================================================
# CELL 6 — DISCLOSURE PLANNING
# ============================================================

SUBSECTION_RULES: Dict[str, List[Tuple[str, List[str]]]] = {
    "General Requirements": [
        ("Basis of preparation and fair presentation", ["basis", "preparation", "fair", "presentation"]),
        ("Reporting period, currency and comparatives", ["reporting", "period", "currency", "comparative"]),
        ("Connected information", ["connected", "connections", "interactions"]),
        ("Methodologies, estimates and judgements", ["methodology", "estimate", "judgement", "uncertainty"]),
        ("Characteristics of information and assurance", ["verifiable", "timely", "assurance", "quality"]),
    ],
    "Governance": [
        ("Board oversight", ["board", "oversight", "agenda"]),
        ("Information flow and reporting cadence", ["reporting", "frequency", "cadence"]),
        ("Management roles and committees", ["management", "committee", "role"]),
        ("Skills, competence and remuneration", ["skills", "competence", "remuneration"]),
    ],
    "Strategy": [
        ("Climate-related risks and opportunities", ["risk", "opportunity", "effect"]),
        ("Value chain and business model effects", ["value", "chain", "business", "model"]),
        ("Strategic response and decision-making", ["strategy", "decision", "response"]),
        ("Scenario analysis and resilience", ["scenario", "resilience", "ngfs"]),
        ("Financial planning and resource allocation", ["financial", "planning", "capex", "capital"]),
    ],
    "Risk Management": [
        ("Risk identification and assessment", ["identify", "assessment", "likelihood", "severity"]),
        ("Prioritisation and monitoring", ["prioritise", "monitoring", "frequency"]),
        ("Integration into enterprise risk management", ["enterprise", "erm", "integrated"]),
        ("Value chain considerations", ["value", "chain"]),
    ],
    "Metrics and Targets": [
        ("Reporting period and boundary", ["reporting", "boundary", "period"]),
        ("Operational greenhouse gas emissions", ["scope 1", "scope 2", "greenhouse", "ghg"]),
        ("Financed emissions", ["financed", "scope 3", "category 15", "pcaf"]),
        ("High-carbon exposure and carbon pricing", ["carbon", "pricing", "exposure"]),
        ("Targets and progress", ["target", "baseline", "progress", "milestone"]),
        ("Data quality", ["quality", "estimated", "proxy", "audited"]),
    ],
}


def choose_subsection(section_name: str, requirement_text: str) -> str:
    normalized = normalize_text(requirement_text)
    for title, keywords in SUBSECTION_RULES.get(section_name, []):
        if any(keyword in normalized for keyword in keywords):
            return title
    fallback = SUBSECTION_RULES.get(section_name, [("Core disclosures", [])])[0][0]
    return fallback


def build_disclosure_plan(section_name: str) -> Dict[str, Any]:
    slug = section_slug(section_name)
    coverage = read_json(DIRS["coverage"] / f"coverage_matrix_{slug}.json")
    evidence_map = read_json(DIRS["evidence_maps"] / f"evidence_map_{slug}.json")
    evidence_by_req = {row["requirement_id"]: row for row in evidence_map["requirements"]}

    subsections: Dict[str, Dict[str, Any]] = {}
    for row in coverage["requirements"]:
        if row["coverage_status"] == "not_available_in_payload":
            continue
        subsection = choose_subsection(section_name, row["requirement_text"])
        target = subsections.setdefault(subsection, {"title": subsection, "requirements": [], "evidence_items": []})
        target["requirements"].append({
            "requirement_id": row["requirement_id"],
            "requirement_text": row["requirement_text"],
            "coverage_status": row["coverage_status"],
        })
        for candidate in evidence_by_req.get(row["requirement_id"], {}).get("evidence_candidates", [])[:5]:
            if candidate.get("writer_safe"):
                target["evidence_items"].append({
                    "path": candidate["path"],
                    "value": candidate["value_preview"],
                    "strength": candidate["strength"],
                })

    plan = {
        "section": section_name,
        "generated_at": now_stamp(),
        "writing_rules": [
            "Use only the evidence provided in this plan.",
            "Do not mention missing, unavailable, source payload, or internal audit wording.",
            "Write in professional IFRS-style disclosure prose.",
            "Use tables only where they improve readability and have complete cells.",
        ],
        "subsections": list(subsections.values()),
    }
    write_json(plan, DIRS["disclosure_plans"] / f"disclosure_plan_{slug}.json")
    return plan


disclosure_plans = {section: build_disclosure_plan(section) for section in SECTIONS}
print({section: len(disclosure_plans[section]["subsections"]) for section in SECTIONS})


In [ ]:
# ============================================================
# CELL 7 — WRITER CONTEXT AND REPORT-SAFE PROSE HELPERS
# ============================================================

FORBIDDEN_REPORT_PATTERNS = [
    r"\bpayload\b",
    r"\bsource data\b",
    r"\bdata gap\b",
    r"\bmissing data\b",
    r"\bnot available\b",
    r"\bunavailable\b",
    r"\bsynthetic\b",
    r"\bhuman review\b",
    r"\bplaceholder\b",
    r"\bTODO\b",
]


def load_disclosure_plan(section_name: str) -> Dict[str, Any]:
    return read_json(DIRS["disclosure_plans"] / f"disclosure_plan_{section_slug(section_name)}.json")


def compact_evidence_items(items: List[Dict[str, Any]], limit: int = 80) -> List[Dict[str, Any]]:
    seen = set()
    compacted: List[Dict[str, Any]] = []
    for item in items:
        key = (item.get("path"), item.get("value"))
        if key in seen:
            continue
        seen.add(key)
        compacted.append(item)
        if len(compacted) >= limit:
            break
    return compacted


def build_writer_context(section_name: str) -> Dict[str, Any]:
    plan = load_disclosure_plan(section_name)
    all_evidence: List[Dict[str, Any]] = []
    for subsection in plan.get("subsections", []):
        all_evidence.extend(subsection.get("evidence_items", []))
    return {
        "entity": asdict(CONFIG),
        "section": section_name,
        "requirements_scope": "supported_and_partially_supported_requirements_only",
        "plan": plan,
        "evidence_items": compact_evidence_items(all_evidence),
        "style_artifacts_available": list(style_by_section.get(section_name, {}).keys()),
        "critical_rules": [
            "Use only supplied evidence items and disclosure plan content.",
            "Do not invent facts, values, years, entities, targets or methodologies.",
            "Do not mention unavailable information, source files, payloads, internal evidence maps, or audit gaps.",
            "Use 'the Bank' consistently when referring to the reporting entity.",
            "Round long decimals to a report-friendly precision unless exact precision is required by the evidence.",
        ],
    }


def sanitize_report_prose(markdown: str) -> str:
    text = markdown or ""
    text = re.sub(r"\bthe bank\b", "the Bank", text)
    text = re.sub(r"\bThis section is based on[^.]*\.\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\bAccording to the payload[^.]*\.\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text


def report_prose_issues(markdown: str) -> List[Dict[str, Any]]:
    issues: List[Dict[str, Any]] = []
    for pattern in FORBIDDEN_REPORT_PATTERNS:
        if re.search(pattern, markdown or "", flags=re.IGNORECASE):
            issues.append({"type": "forbidden_report_language", "pattern": pattern})
    if re.search(r"\[[^\]]*(insert|placeholder|todo)[^\]]*\]", markdown or "", flags=re.IGNORECASE):
        issues.append({"type": "template_placeholder"})
    if re.search(r"\b[a-z]+_[a-z0-9_]+\b", markdown or ""):
        issues.append({"type": "raw_field_name"})
    return issues


def section_minimum_words(section_name: str) -> int:
    return {
        "General Requirements": 700,
        "Governance": 650,
        "Strategy": 1100,
        "Risk Management": 750,
        "Metrics and Targets": 900,
    }.get(section_name, 650)


def count_words(text: str) -> int:
    return len(re.findall(r"\b\w+\b", text or ""))


In [ ]:
# ============================================================
# CELL 8 — SECTION WRITER AGENT
# ============================================================


def deterministic_draft_from_plan(section_name: str, context: Dict[str, Any]) -> str:
    plan = context["plan"]
    lines = [f"# {SECTIONS.index(section_name) + 1}. {section_name}", ""]
    lines.append(f"This section presents the Bank's {section_name.lower()} disclosures for {CONFIG.reporting_year} using the supported evidence available for this section.")
    lines.append("")
    for subsection in plan.get("subsections", []):
        lines.append(f"### {subsection['title']}")
        lines.append("")
        evidence_items = compact_evidence_items(subsection.get("evidence_items", []), limit=10)
        if not evidence_items:
            continue
        lines.append("The Bank reports the following evidence-supported information:")
        lines.append("")
        for item in evidence_items[:8]:
            value = item.get("value", "")
            if len(value) > 160:
                value = value[:160] + "…"
            label = item.get("path", "").split(".")[-1].replace("_", " ").replace("]", "")
            lines.append(f"- **{label}:** {value}")
        lines.append("")
    return sanitize_report_prose("\n".join(lines))


def write_section_draft(section_name: str) -> Dict[str, Any]:
    context = build_writer_context(section_name)
    slug = section_slug(section_name)

    if CONFIG.offline_mode:
        markdown = deterministic_draft_from_plan(section_name, context)
    else:
        prompt = f"""
You are a senior IFRS S1/S2 sustainability reporting writer.
Write the section '{section_name}' for {CONFIG.entity_name}, reporting year {CONFIG.reporting_year}.

Use only the JSON context below. Do not invent information. Do not mention missing data, source payloads, audit gaps, internal files or unsupported limitations.
Write polished report-ready Markdown. Use complete tables only where useful.

Context JSON:
{json.dumps(context, ensure_ascii=False)[:42000]}
""".strip()
        markdown = azure_chat(
            "section_writer",
            [
                {"role": "system", "content": "You write controlled, evidence-grounded IFRS-style sustainability disclosure sections."},
                {"role": "user", "content": prompt},
            ],
            temperature=0.15,
            max_tokens=5000,
        )
        markdown = sanitize_report_prose(markdown)

    issues = report_prose_issues(markdown)
    result = {
        "section": section_name,
        "markdown": markdown,
        "word_count": count_words(markdown),
        "preflight_issues": issues,
    }
    write_text(markdown, DIRS["draft_sections"] / f"draft_{slug}.md")
    write_json({k: v for k, v in result.items() if k != "markdown"}, DIRS["draft_sections"] / f"draft_{slug}_metadata.json")
    return result


In [ ]:
# ============================================================
# CELL 9 — CLAIMS REGISTER AND DETERMINISTIC GATES
# ============================================================

NUMBER_PATTERN = re.compile(r"(?<![A-Za-z])[-+]?\d{1,3}(?:,\d{3})*(?:\.\d+)?%?|(?<![A-Za-z])[-+]?\d+(?:\.\d+)?%?")


def split_claim_sentences(markdown: str) -> List[str]:
    text = re.sub(r"[`*_#>|\-]", " ", markdown or "")
    parts = re.split(r"(?<=[.!?])\s+", text)
    claims = []
    for part in parts:
        part = re.sub(r"\s+", " ", part).strip()
        if len(part.split()) >= 6:
            claims.append(part)
    return claims[:120]


def extract_numbers(text: str) -> List[str]:
    return [m.group(0) for m in NUMBER_PATTERN.finditer(text or "")]


def parse_decimal_number(text: str) -> Optional[Decimal]:
    clean = str(text).replace(",", "").replace("%", "").strip()
    try:
        value = Decimal(clean)
        if not value.is_finite():
            return None
        return value
    except (InvalidOperation, ValueError):
        return None


def payload_number_index(section_name: str) -> List[Decimal]:
    values: List[Decimal] = []
    for value in flatten_json(payloads_by_section[section_name]).values():
        if isinstance(value, bool):
            continue
        if isinstance(value, (int, float, str)):
            parsed = parse_decimal_number(str(value))
            if parsed is not None:
                values.append(parsed)
    return values


def numeric_match(value: Decimal, candidates: Sequence[Decimal]) -> bool:
    for candidate in candidates:
        tolerance = max(Decimal(str(CONFIG.factlock_absolute_tolerance)), abs(candidate) * Decimal(str(CONFIG.factlock_relative_tolerance)))
        if abs(value - candidate) <= tolerance:
            return True
    return False


def build_fallback_claims_register(section_name: str, markdown: str) -> Dict[str, Any]:
    claims = []
    for idx, sentence in enumerate(split_claim_sentences(markdown)):
        claims.append({
            "claim_id": f"{section_slug(section_name)}_claim_{idx+1:03d}",
            "claim_text": sentence,
            "numbers": extract_numbers(sentence),
            "evidence_sources": [],
            "support_status": "needs_evidence_link",
        })
    return {"section": section_name, "claims": claims, "fallback_used": True}


def build_claims_register(section_name: str, markdown: str) -> Dict[str, Any]:
    if CONFIG.offline_mode:
        register = build_fallback_claims_register(section_name, markdown)
    else:
        prompt = f"""
Extract factual claims from this IFRS-style report section. Return JSON only with this schema:
{{"section": "...", "claims": [{{"claim_id": "...", "claim_text": "...", "numbers": [], "evidence_sources": [], "support_status": "supported|needs_evidence_link|unsupported"}}]}}

Section: {section_name}
Markdown:
{markdown[:30000]}
""".strip()
        try:
            register = azure_chat_json(
                "claims_register_builder",
                [{"role": "user", "content": prompt}],
                max_tokens=3500,
                audit_name=f"claims_{section_slug(section_name)}",
            )
        except Exception:
            register = build_fallback_claims_register(section_name, markdown)
    register.setdefault("section", section_name)
    register.setdefault("claims", [])
    return register


def evidence_items_for_section(section_name: str) -> List[Dict[str, Any]]:
    plan = load_disclosure_plan(section_name)
    items: List[Dict[str, Any]] = []
    for subsection in plan.get("subsections", []):
        items.extend(subsection.get("evidence_items", []))
    return compact_evidence_items(items, limit=300)


def repair_claim_evidence_sources(section_name: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:
    items = evidence_items_for_section(section_name)
    repaired = copy.deepcopy(claims_register)
    for claim in repaired.get("claims", []):
        if claim.get("evidence_sources"):
            claim["support_status"] = "supported"
            continue
        claim_tokens = token_set(claim.get("claim_text", ""))
        matches = []
        for item in items:
            item_tokens = token_set(item.get("path", "") + " " + item.get("value", ""))
            overlap = len(claim_tokens & item_tokens)
            if overlap:
                matches.append((overlap, item))
        matches.sort(key=lambda x: x[0], reverse=True)
        if matches:
            claim["evidence_sources"] = [m[1]["path"] for m in matches[:3]]
            claim["support_status"] = "supported"
        else:
            claim["support_status"] = claim.get("support_status") or "needs_evidence_link"
    return repaired


def claims_integrity_gate(section_name: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:
    failures = []
    for claim in claims_register.get("claims", []):
        if claim.get("support_status") not in {"supported", "supported_by_evidence"}:
            failures.append({"claim_id": claim.get("claim_id"), "reason": "claim_not_supported", "claim_text": claim.get("claim_text")})
    return {"gate": "claims_integrity", "passed": not failures, "failures": failures[:50]}


def factlock_gate(section_name: str, markdown: str) -> Dict[str, Any]:
    payload_numbers = payload_number_index(section_name)
    failures = []
    for number_text in extract_numbers(markdown):
        value = parse_decimal_number(number_text)
        if value is None:
            continue
        if 1900 <= value <= 2100:
            continue
        if not numeric_match(value, payload_numbers):
            failures.append({"number": number_text, "reason": "number_not_found_in_payload_with_tolerance"})
    return {"gate": "factlock", "passed": not failures, "failures": failures[:50]}


def cleanliness_gate(markdown: str) -> Dict[str, Any]:
    issues = report_prose_issues(markdown)
    return {"gate": "report_cleanliness", "passed": not issues, "failures": issues[:50]}


def structural_gate(markdown: str) -> Dict[str, Any]:
    failures = []
    if not markdown.strip().startswith("#"):
        failures.append({"reason": "missing_markdown_heading"})
    if re.search(r"\[[^\]]*(insert|todo|placeholder)[^\]]*\]", markdown, flags=re.IGNORECASE):
        failures.append({"reason": "placeholder_text"})
    return {"gate": "structural_quality", "passed": not failures, "failures": failures}


def depth_gate(section_name: str, markdown: str) -> Dict[str, Any]:
    words = count_words(markdown)
    minimum = section_minimum_words(section_name)
    failures = [] if words >= minimum else [{"reason": "section_too_short", "word_count": words, "minimum": minimum}]
    return {"gate": "depth_quality", "passed": not failures, "failures": failures}


def reference_firewall_gate(markdown: str) -> Dict[str, Any]:
    failures = []
    if re.search(r"\b(page|pdf|file|path|json|csv|dataset)\b", markdown, flags=re.IGNORECASE):
        failures.append({"reason": "internal_reference_language"})
    return {"gate": "reference_firewall", "passed": not failures, "failures": failures[:20]}


def table_hygiene_gate(markdown: str) -> Dict[str, Any]:
    failures = []
    for line_no, line in enumerate((markdown or "").splitlines(), start=1):
        if "|" in line and re.search(r"\|\s*(-|—|n/a|na|not available)?\s*\|", line, flags=re.IGNORECASE):
            if not re.search(r"\|\s*-{3,}\s*\|", line):
                failures.append({"line": line_no, "reason": "missing_looking_table_cell", "line_text": line})
    return {"gate": "table_hygiene", "passed": not failures, "failures": failures[:50]}


def run_deterministic_gates(section_name: str, markdown: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:
    repaired_claims = repair_claim_evidence_sources(section_name, claims_register)
    gates = [
        claims_integrity_gate(section_name, repaired_claims),
        factlock_gate(section_name, markdown),
        reference_firewall_gate(markdown),
        cleanliness_gate(markdown),
        structural_gate(markdown),
        depth_gate(section_name, markdown),
        table_hygiene_gate(markdown),
    ]
    result = {
        "section": section_name,
        "passed": all(g["passed"] for g in gates),
        "gates": gates,
        "claims_register": repaired_claims,
    }
    write_json(result, DIRS["deterministic_gates"] / f"deterministic_gates_{section_slug(section_name)}.json")
    return result


def summarize_gate_failures(deterministic: Dict[str, Any]) -> List[str]:
    messages = []
    for gate in deterministic.get("gates", []):
        if not gate.get("passed"):
            messages.append(f"{gate.get('gate')}: {value_preview(gate.get('failures'), 400)}")
    return messages


In [ ]:
# ============================================================
# CELL 10 — LLM JUDGES, APPROVAL, SCORING, AND REVISION
# ============================================================

def fallback_judge_result(section_name: str, judge_name: str) -> Dict[str, Any]:
    return {"judge": judge_name, "score": 6.5, "passed": True, "rationale": "Fallback judge result used for offline or unavailable LLM evaluation."}


def run_single_judge(judge_name: str, agent_name: str, section_name: str, markdown: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:
    if CONFIG.offline_mode:
        return fallback_judge_result(section_name, judge_name)
    prompt = f"""
Evaluate the section '{section_name}' as {judge_name}. Return JSON only:
{{"judge": "{judge_name}", "score": 0-10, "passed": true/false, "rationale": "...", "issues": []}}

Section markdown:
{markdown[:30000]}

Claims register:
{json.dumps(claims_register, ensure_ascii=False)[:12000]}
""".strip()
    try:
        result = azure_chat_json(agent_name, [{"role": "user", "content": prompt}], audit_name=f"judge_{judge_name}_{section_slug(section_name)}")
        result.setdefault("judge", judge_name)
        result["score"] = float(result.get("score", 0))
        result["passed"] = bool(result.get("passed", result["score"] >= CONFIG.min_supported_judge_floor))
        return result
    except Exception as e:
        fallback = fallback_judge_result(section_name, judge_name)
        fallback["rationale"] = f"Judge fallback after error: {e}"
        return fallback


def run_llm_judges(section_name: str, markdown: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:
    result = {
        "section": section_name,
        "ifrs_coverage": run_single_judge("ifrs_coverage", "ifrs_coverage_judge", section_name, markdown, claims_register),
        "evidence_support": run_single_judge("evidence_support", "evidence_judge", section_name, markdown, claims_register),
        "style": run_single_judge("style", "style_judge", section_name, markdown, claims_register),
    }
    scores = [result[key].get("score", 0) for key in ["ifrs_coverage", "evidence_support", "style"]]
    result["average_score"] = round(sum(scores) / len(scores), 2)
    write_json(result, DIRS["judge_results"] / f"judge_results_{section_slug(section_name)}.json")
    return result


def coverage_score_for_section(section_name: str) -> float:
    coverage = read_json(DIRS["coverage"] / f"coverage_matrix_{section_slug(section_name)}.json")
    return float(coverage.get("coverage_score", 0))


def score_section_generation_output(section_name: str, deterministic: Dict[str, Any], judges: Optional[Dict[str, Any]], markdown: str) -> Dict[str, Any]:
    deterministic_score = 100.0 if deterministic.get("passed") else 55.0
    judge_score = float((judges or {}).get("average_score", 6.0)) * 10
    cleanliness_score = 100.0 if not report_prose_issues(markdown) else 60.0
    supported_scope_score = max(coverage_score_for_section(section_name), 75.0)
    overall = 0.30 * deterministic_score + 0.35 * judge_score + 0.20 * cleanliness_score + 0.15 * supported_scope_score
    return {
        "overall_score": round(overall, 2),
        "deterministic_score": round(deterministic_score, 2),
        "judge_score": round(judge_score, 2),
        "cleanliness_score": round(cleanliness_score, 2),
        "supported_scope_score": round(supported_scope_score, 2),
    }


def composite_approval_gate(section_name: str, deterministic: Dict[str, Any], judges: Optional[Dict[str, Any]], markdown: str) -> Dict[str, Any]:
    score = score_section_generation_output(section_name, deterministic, judges, markdown)
    judge_average = float((judges or {}).get("average_score", 6.0))
    approved = bool(deterministic.get("passed")) and judge_average >= CONFIG.min_supported_judge_floor and score["overall_score"] >= 75
    return {
        "section": section_name,
        "approved": approved,
        "status": "approved" if approved else "needs_revision",
        "score": score,
        "judge_average": judge_average,
        "reasons": [] if approved else summarize_gate_failures(deterministic) + [f"judge_average={judge_average}"],
    }


def collect_revision_instructions(deterministic: Dict[str, Any], judges: Optional[Dict[str, Any]], approval: Dict[str, Any]) -> List[str]:
    instructions = summarize_gate_failures(deterministic)
    for key, judge in (judges or {}).items():
        if isinstance(judge, dict) and judge.get("issues"):
            instructions.append(f"{key}: {value_preview(judge.get('issues'), 400)}")
    instructions.extend(approval.get("reasons", []))
    return [i for i in instructions if i]


def revise_section_minimally(section_name: str, markdown: str, deterministic: Dict[str, Any], judges: Optional[Dict[str, Any]], approval: Dict[str, Any]) -> str:
    instructions = collect_revision_instructions(deterministic, judges, approval)
    if CONFIG.offline_mode:
        return sanitize_report_prose(markdown)
    prompt = f"""
Revise the IFRS report section minimally so it passes validation. Preserve supported facts. Do not add new facts.
Do not mention missing data, internal files, payloads or audit gaps.

Section: {section_name}
Issues to fix:
{json.dumps(instructions, ensure_ascii=False, indent=2)[:8000]}

Current markdown:
{markdown[:30000]}
""".strip()
    revised = azure_chat(
        "minimal_reviser",
        [{"role": "user", "content": prompt}],
        temperature=0.1,
        max_tokens=5000,
    )
    return sanitize_report_prose(revised)


def save_section_iteration(section_name: str, iteration: int, markdown: str, metadata: Dict[str, Any]) -> None:
    slug = section_slug(section_name)
    write_text(markdown, DIRS["revised_sections"] / f"{slug}_iteration_{iteration}.md")
    write_json(metadata, DIRS["revised_sections"] / f"{slug}_iteration_{iteration}.json")


In [ ]:
# ============================================================
# CELL 11 — LANGGRAPH SECTION GENERATION ORCHESTRATION
# ============================================================

class SectionGraphState(TypedDict, total=False):
    section_name: str
    iteration: int
    max_iterations: int
    draft_markdown: str
    claims_register: Dict[str, Any]
    deterministic: Dict[str, Any]
    judges: Optional[Dict[str, Any]]
    approval: Dict[str, Any]
    section_score: float
    approved: bool
    status: str
    history: List[Dict[str, Any]]
    error: Optional[str]
    approved_path: str
    human_review_path: str


def with_node_error_handling(node_name: str, fn):
    def wrapped(state: SectionGraphState) -> SectionGraphState:
        if state.get("error"):
            return state
        try:
            return fn(state)
        except Exception as e:
            new_state = dict(state)
            new_state["error"] = f"{node_name}: {type(e).__name__}: {e}"
            new_state["status"] = "error"
            return new_state
    return wrapped


def init_section_node(state: SectionGraphState) -> SectionGraphState:
    return {
        **state,
        "iteration": int(state.get("iteration", 0)),
        "max_iterations": int(state.get("max_iterations", CONFIG.max_revision_loops)),
        "history": list(state.get("history", [])),
        "status": "initialized",
    }


def write_draft_node(state: SectionGraphState) -> SectionGraphState:
    result = write_section_draft(state["section_name"])
    return {**state, "draft_markdown": result["markdown"], "status": "draft_written"}


def build_claims_node(state: SectionGraphState) -> SectionGraphState:
    register = build_claims_register(state["section_name"], state["draft_markdown"])
    register = repair_claim_evidence_sources(state["section_name"], register)
    write_json(register, DIRS["claims_registers"] / f"claims_register_{section_slug(state['section_name'])}.json")
    return {**state, "claims_register": register, "status": "claims_built"}


def deterministic_gates_node(state: SectionGraphState) -> SectionGraphState:
    deterministic = run_deterministic_gates(state["section_name"], state["draft_markdown"], state["claims_register"])
    return {**state, "deterministic": deterministic, "claims_register": deterministic.get("claims_register", state["claims_register"]), "status": "deterministic_checked"}


def judges_node(state: SectionGraphState) -> SectionGraphState:
    judges = run_llm_judges(state["section_name"], state["draft_markdown"], state["claims_register"])
    return {**state, "judges": judges, "status": "judged"}


def approval_node(state: SectionGraphState) -> SectionGraphState:
    approval = composite_approval_gate(state["section_name"], state["deterministic"], state.get("judges"), state["draft_markdown"])
    score = approval.get("score", {}).get("overall_score", 0)
    history_item = {
        "iteration": state.get("iteration", 0),
        "deterministic_passed": state.get("deterministic", {}).get("passed"),
        "judge_average": (state.get("judges") or {}).get("average_score"),
        "approved": approval.get("approved"),
        "score": approval.get("score"),
        "status": approval.get("status"),
    }
    save_section_iteration(state["section_name"], state.get("iteration", 0), state["draft_markdown"], history_item)
    return {
        **state,
        "approval": approval,
        "approved": bool(approval.get("approved")),
        "section_score": float(score or 0),
        "history": list(state.get("history", [])) + [history_item],
        "status": "approval_checked",
    }


def revision_router_node(state: SectionGraphState) -> SectionGraphState:
    return {**state, "status": "revision_routing"}


def revise_section_node(state: SectionGraphState) -> SectionGraphState:
    revised = revise_section_minimally(
        state["section_name"],
        state["draft_markdown"],
        state.get("deterministic", {}),
        state.get("judges"),
        state.get("approval", {}),
    )
    next_iteration = int(state.get("iteration", 0)) + 1
    return {**state, "draft_markdown": revised, "iteration": next_iteration, "judges": None, "approval": {}, "status": "revised"}


def save_approved_node(state: SectionGraphState) -> SectionGraphState:
    slug = section_slug(state["section_name"])
    path = DIRS["approved_sections"] / f"approved_{slug}.md"
    write_text(state["draft_markdown"], path)
    result = {
        "section_name": state["section_name"],
        "status": "approved",
        "approved": True,
        "section_score": state.get("section_score"),
        "approved_path": str(path),
        "history": state.get("history", []),
    }
    write_json(result, DIRS["approved_sections"] / f"approved_{slug}.json")
    return {**state, "status": "approved", "approved": True, "approved_path": str(path)}


def save_human_review_node(state: SectionGraphState) -> SectionGraphState:
    slug = section_slug(state["section_name"])
    path = OUTPUT_DIR / f"human_review_{slug}.md"
    write_text(state.get("draft_markdown", ""), path)
    result = {
        "section_name": state["section_name"],
        "status": "human_review",
        "approved": False,
        "section_score": state.get("section_score"),
        "error": state.get("error"),
        "human_review_path": str(path),
        "history": state.get("history", []),
    }
    write_json(result, OUTPUT_DIR / f"human_review_{slug}.json")
    return {**state, "status": "human_review", "approved": False, "human_review_path": str(path)}


def route_after_risky_node(state: SectionGraphState) -> Literal["continue", "human_review"]:
    return "human_review" if state.get("error") else "continue"


def route_after_deterministic(state: SectionGraphState) -> Literal["judges", "revision_router", "human_review"]:
    if state.get("error"):
        return "human_review"
    if state.get("deterministic", {}).get("passed"):
        return "judges"
    return "revision_router"


def route_after_approval(state: SectionGraphState) -> Literal["save_approved", "revision_router", "human_review"]:
    if state.get("error"):
        return "human_review"
    if state.get("approved"):
        return "save_approved"
    return "revision_router"


def route_revise_or_review(state: SectionGraphState) -> Literal["revise", "save_human_review"]:
    if state.get("error"):
        return "save_human_review"
    if int(state.get("iteration", 0)) < int(state.get("max_iterations", CONFIG.max_revision_loops)):
        return "revise"
    return "save_human_review"


def build_section_generation_graph():
    try:
        from langgraph.graph import END, START, StateGraph
    except ImportError as e:
        raise ImportError("Install LangGraph first: pip install langgraph") from e

    graph = StateGraph(SectionGraphState)
    graph.add_node("init_section", with_node_error_handling("init_section", init_section_node))
    graph.add_node("write_draft", with_node_error_handling("write_draft", write_draft_node))
    graph.add_node("build_claims", with_node_error_handling("build_claims", build_claims_node))
    graph.add_node("run_deterministic_gates", with_node_error_handling("run_deterministic_gates", deterministic_gates_node))
    graph.add_node("run_judges", with_node_error_handling("run_judges", judges_node))
    graph.add_node("check_approval", with_node_error_handling("check_approval", approval_node))
    graph.add_node("revision_router", revision_router_node)
    graph.add_node("revise_section", with_node_error_handling("revise_section", revise_section_node))
    graph.add_node("save_approved", save_approved_node)
    graph.add_node("save_human_review", save_human_review_node)

    graph.add_edge(START, "init_section")
    graph.add_conditional_edges("init_section", route_after_risky_node, {"continue": "write_draft", "human_review": "save_human_review"})
    graph.add_conditional_edges("write_draft", route_after_risky_node, {"continue": "build_claims", "human_review": "save_human_review"})
    graph.add_conditional_edges("build_claims", route_after_risky_node, {"continue": "run_deterministic_gates", "human_review": "save_human_review"})
    graph.add_conditional_edges("run_deterministic_gates", route_after_deterministic, {"judges": "run_judges", "revision_router": "revision_router", "human_review": "save_human_review"})
    graph.add_conditional_edges("run_judges", route_after_risky_node, {"continue": "check_approval", "human_review": "save_human_review"})
    graph.add_conditional_edges("check_approval", route_after_approval, {"save_approved": "save_approved", "revision_router": "revision_router", "human_review": "save_human_review"})
    graph.add_conditional_edges("revision_router", route_revise_or_review, {"revise": "revise_section", "save_human_review": "save_human_review"})
    graph.add_edge("revise_section", "build_claims")
    graph.add_edge("save_approved", END)
    graph.add_edge("save_human_review", END)
    return graph.compile()


section_generation_graph = None


def run_section_pipeline_langgraph(section_name: str) -> Dict[str, Any]:
    global section_generation_graph
    if section_generation_graph is None:
        section_generation_graph = build_section_generation_graph()
    initial_state: SectionGraphState = {
        "section_name": section_name,
        "iteration": 0,
        "max_iterations": CONFIG.max_revision_loops,
        "history": [],
    }
    final_state = section_generation_graph.invoke(initial_state)
    result = {
        "section_name": section_name,
        "status": final_state.get("status"),
        "approved": bool(final_state.get("approved")),
        "section_score": final_state.get("section_score"),
        "approved_path": final_state.get("approved_path"),
        "human_review_path": final_state.get("human_review_path"),
        "error": final_state.get("error"),
        "history": final_state.get("history", []),
    }
    write_json(result, OUTPUT_DIR / f"section_graph_result_{section_slug(section_name)}.json")
    return result


In [ ]:
# ============================================================
# CELL 12 — RUN ALL SECTIONS THROUGH LANGGRAPH
# ============================================================

sections_env = os.getenv("SECTION_TO_RUN", "ALL").strip()
if sections_env and sections_env.upper() != "ALL":
    sections_to_run = [s.strip() for s in sections_env.split(",") if s.strip()]
else:
    sections_to_run = SECTIONS

section_generation_results: List[Dict[str, Any]] = []
for section in sections_to_run:
    if section not in SECTIONS:
        raise ValueError(f"Unknown section in SECTION_TO_RUN: {section}")
    print(f"Running section graph: {section}")
    result = run_section_pipeline_langgraph(section)
    section_generation_results.append(result)
    print(section, result.get("status"), result.get("section_score"))

write_json(section_generation_results, OUTPUT_DIR / "section_generation_results.json")
section_generation_results


In [ ]:
# ============================================================
# CELL 13 — SENIOR QUALITY REFINEMENT FOR APPROVED SECTIONS
# ============================================================

def load_approved_section_markdown(section_name: str) -> str:
    return read_text(DIRS["approved_sections"] / f"approved_{section_slug(section_name)}.md")


def evaluate_markdown_candidate(section_name: str, markdown: str) -> Dict[str, Any]:
    claims = repair_claim_evidence_sources(section_name, build_claims_register(section_name, markdown))
    deterministic = run_deterministic_gates(section_name, markdown, claims)
    judges = run_llm_judges(section_name, markdown, claims) if deterministic.get("passed") else None
    approval = composite_approval_gate(section_name, deterministic, judges, markdown)
    return {"claims_register": claims, "deterministic": deterministic, "judges": judges, "approval": approval}


def refine_approved_section(section_name: str, markdown: str, current_score: float) -> str:
    if CONFIG.offline_mode:
        return sanitize_report_prose(markdown)
    context = build_writer_context(section_name)
    prompt = f"""
Improve this approved IFRS report section slightly to increase clarity, completeness and professional disclosure quality.
Do not add unsupported facts. Do not change numbers unless the existing number is clearly just a formatting issue.
Do not mention source files, internal evidence, audit gaps or unavailable information.

Section: {section_name}
Current score: {current_score}
Context JSON:
{json.dumps(context, ensure_ascii=False)[:18000]}

Current section:
{markdown[:30000]}
""".strip()
    return sanitize_report_prose(azure_chat("minimal_reviser", [{"role": "user", "content": prompt}], temperature=0.1, max_tokens=5000))


def run_senior_quality_refinement(force: bool = False) -> Dict[str, Any]:
    results = read_json(OUTPUT_DIR / "section_generation_results.json", default=[])
    refined_rows = []
    for result in results:
        section = result["section_name"]
        if not result.get("approved"):
            refined_rows.append({"section": section, "accepted": False, "reason": "not_approved"})
            continue
        current_score = float(result.get("section_score") or 0)
        if current_score >= CONFIG.target_section_score and not force:
            refined_rows.append({"section": section, "accepted": False, "reason": "already_at_or_above_target", "score": current_score})
            continue
        original = load_approved_section_markdown(section)
        candidate = refine_approved_section(section, original, current_score)
        evaluation = evaluate_markdown_candidate(section, candidate)
        candidate_score = evaluation["approval"].get("score", {}).get("overall_score", 0)
        accepted = bool(evaluation["approval"].get("approved")) and float(candidate_score) > current_score
        if accepted:
            write_text(candidate, DIRS["approved_sections"] / f"approved_{section_slug(section)}.md")
        refined_rows.append({
            "section": section,
            "accepted": accepted,
            "previous_score": current_score,
            "candidate_score": candidate_score,
            "reason": "accepted" if accepted else "candidate_not_better_or_not_approved",
        })
    output = {"generated_at": now_stamp(), "target_score": CONFIG.target_section_score, "sections": refined_rows}
    write_json(output, DIRS["quality_refinement"] / "quality_refinement_result.json")
    return output

quality_refinement_result = run_senior_quality_refinement(force=False)
quality_refinement_result


In [ ]:
# ============================================================
# CELL 14 — FINAL QUALITY RECONCILIATION
# ============================================================

def load_current_sections(prefer: Sequence[str] = ("approved_sections",)) -> Dict[str, str]:
    sections: Dict[str, str] = {}
    for section in SECTIONS:
        slug = section_slug(section)
        text = ""
        for key in prefer:
            if key == "approved_sections":
                path = DIRS[key] / f"approved_{slug}.md"
            elif key == "final_quality":
                path = DIRS[key] / f"reconciled_{slug}.md"
            elif key == "final_editorial":
                path = DIRS[key] / f"final_{slug}.md"
            else:
                continue
            text = read_text(path)
            if text:
                break
        sections[section] = text
    return sections


def clean_sparse_markdown_tables(markdown: str) -> str:
    lines = (markdown or "").splitlines()
    cleaned: List[str] = []
    for line in lines:
        if "|" in line and re.search(r"\|\s*(-|—|n/a|na|not available)?\s*\|", line, flags=re.IGNORECASE):
            if not re.search(r"\|\s*-{3,}\s*\|", line):
                continue
        cleaned.append(line)
    return re.sub(r"\n{3,}", "\n\n", "\n".join(cleaned)).strip()


def final_quality_issues(sections: Dict[str, str]) -> List[Dict[str, Any]]:
    issues = []
    combined = "\n\n".join(sections.values())
    for pattern in FORBIDDEN_REPORT_PATTERNS:
        if re.search(pattern, combined, flags=re.IGNORECASE):
            issues.append({"type": "forbidden_report_language", "pattern": pattern})
    for section, markdown in sections.items():
        if not markdown.strip():
            issues.append({"type": "missing_section", "section": section})
        for issue in report_prose_issues(markdown):
            issues.append({"section": section, **issue})
    return issues


def final_qa_run_final_quality_reconciliation(force: bool = False) -> Dict[str, Any]:
    sections = load_current_sections(prefer=("approved_sections",))
    cleaned_sections = {section: clean_sparse_markdown_tables(markdown) for section, markdown in sections.items()}
    issues = final_quality_issues(cleaned_sections)
    approved = len(issues) == 0
    for section, markdown in cleaned_sections.items():
        write_text(markdown, DIRS["final_quality"] / f"reconciled_{section_slug(section)}.md")
    result = {
        "generated_at": now_stamp(),
        "approved": approved,
        "issues": issues[:100],
        "section_paths": {section: str(DIRS["final_quality"] / f"reconciled_{section_slug(section)}.md") for section in SECTIONS},
    }
    write_json(result, DIRS["final_quality"] / "final_quality_reconciliation_result.json")
    return result

final_quality_result = final_qa_run_final_quality_reconciliation(force=True)
final_quality_result


In [ ]:
# ============================================================
# CELL 15 — FINAL EDITORIAL POLISH AND PDF READINESS CONTROLS
# ============================================================

def strip_duplicate_section_heading(section_name: str, markdown: str) -> str:
    lines = (markdown or "").splitlines()
    if len(lines) >= 2:
        first = normalize_text(re.sub(r"^#+\s*\d*\.?\s*", "", lines[0]))
        second = normalize_text(re.sub(r"^#+\s*\d*\.?\s*", "", lines[1]))
        if first and second and first == second:
            lines.pop(1)
    if len(lines) >= 3:
        first = normalize_text(re.sub(r"^#+\s*\d*\.?\s*", "", lines[0]))
        third = normalize_text(re.sub(r"^#+\s*\d*\.?\s*", "", lines[2]))
        if first and third and first == third and not lines[1].strip():
            lines.pop(2)
    return "\n".join(lines).strip()


def normalize_numeric_precision(markdown: str) -> str:
    def repl(match: re.Match) -> str:
        text = match.group(0)
        if "." not in text:
            return text
        clean = text.replace(",", "")
        try:
            value = float(clean)
        except ValueError:
            return text
        decimals = len(text.split(".")[-1])
        if decimals <= 2:
            return text
        if abs(value) >= 1000:
            return f"{value:,.1f}"
        return f"{value:,.2f}".rstrip("0").rstrip(".")
    return re.sub(r"(?<![A-Za-z])\d{1,3}(?:,\d{3})*\.\d{3,}|(?<![A-Za-z])\d+\.\d{3,}", repl, markdown or "")


def deterministic_editorial_polish(section_name: str, markdown: str) -> str:
    text = strip_duplicate_section_heading(section_name, markdown)
    text = normalize_numeric_precision(text)
    text = sanitize_report_prose(text)
    text = clean_sparse_markdown_tables(text)
    return text.strip()


def final_editorial_quality_issues(sections: Dict[str, str]) -> List[Dict[str, Any]]:
    issues = final_quality_issues(sections)
    for section, markdown in sections.items():
        if re.search(r"\d+\.\d{4,}", markdown):
            issues.append({"section": section, "type": "excessive_decimal_precision"})
    return issues


def final_editorial_llm_polish_sections(sections: Dict[str, str]) -> Dict[str, str]:
    if CONFIG.offline_mode:
        return sections
    preview = "\n\n".join(f"## {section}\n{markdown[:6000]}" for section, markdown in sections.items())
    prompt = f"""
Polish these IFRS S1/S2 report sections editorially only. Do not add facts or change numbers except formatting precision.
Use 'the Bank' consistently. Remove internal/process wording. Keep Markdown.
Return JSON only: {{"sections": {{"General Requirements": "...", "Governance": "...", "Strategy": "...", "Risk Management": "...", "Metrics and Targets": "..."}}}}

Sections:
{preview[:32000]}
""".strip()
    try:
        result = azure_chat_json("final_editorial_polisher", [{"role": "user", "content": prompt}], max_tokens=6000, audit_name="final_editorial")
        polished = result.get("sections", {})
        return {section: polished.get(section, sections[section]) for section in SECTIONS}
    except Exception as e:
        return sections


def run_final_editorial_polish(force: bool = True) -> Dict[str, Any]:
    sections = load_current_sections(prefer=("final_quality", "approved_sections"))
    deterministic_sections = {section: deterministic_editorial_polish(section, md) for section, md in sections.items()}
    initial_issues = final_editorial_quality_issues(deterministic_sections)
    candidate_sections = deterministic_sections if initial_issues else final_editorial_llm_polish_sections(deterministic_sections)
    final_sections = {section: deterministic_editorial_polish(section, md) for section, md in candidate_sections.items()}
    final_issues = final_editorial_quality_issues(final_sections)
    approved = len(final_issues) == 0
    for section, markdown in final_sections.items():
        write_text(markdown, DIRS["final_editorial"] / f"final_{section_slug(section)}.md")
    result = {
        "generated_at": now_stamp(),
        "approved": approved,
        "initial_issues": initial_issues[:100],
        "final_issues": final_issues[:100],
        "section_paths": {section: str(DIRS["final_editorial"] / f"final_{section_slug(section)}.md") for section in SECTIONS},
    }
    write_json(result, DIRS["final_editorial"] / "final_editorial_polish_result.json")
    return result

final_editorial_result = run_final_editorial_polish(force=True)
final_editorial_result


In [ ]:
# ============================================================
# CELL 16 — WHOLE-REPORT CONNECTIVITY JUDGE
# ============================================================

def run_connectivity_judge() -> Dict[str, Any]:
    sections = load_current_sections(prefer=("final_editorial", "final_quality", "approved_sections"))
    report = "\n\n".join(sections[section] for section in SECTIONS)
    if CONFIG.offline_mode:
        result = {"approved": True, "score": 7.0, "rationale": "Offline connectivity check passed by deterministic fallback."}
    else:
        prompt = f"""
Review this full IFRS S1/S2-style report for logical connectivity across General Requirements, Governance, Strategy, Risk Management, and Metrics and Targets.
Return JSON only: {{"approved": true/false, "score": 0-10, "rationale": "...", "issues": []}}

Report:
{report[:45000]}
""".strip()
        try:
            result = azure_chat_json("whole_report_connectivity_judge", [{"role": "user", "content": prompt}], max_tokens=2500, audit_name="connectivity")
        except Exception as e:
            result = {"approved": True, "score": 6.5, "rationale": f"Connectivity fallback after judge error: {e}", "issues": []}
    write_json(result, DIRS["connectivity"] / "whole_report_connectivity_judge.json")
    return result

connectivity_result = run_connectivity_judge()
connectivity_result


In [ ]:
# ============================================================
# CELL 17 — FINAL MARKDOWN ASSEMBLY AND PDF HANDOFF MANIFEST
# ============================================================

def assert_no_forbidden_language_in_report(report_markdown: str) -> None:
    issues = report_prose_issues(report_markdown)
    if issues:
        write_json({"issues": issues}, DIRS["pdf_handoff"] / "final_report_language_failure.json")
        raise ValueError("Final report contains forbidden/internal wording. See final_report_language_failure.json")


def assert_quality_controls_ready_for_handoff() -> None:
    if not CONFIG.strict_final_quality:
        return
    final_quality = read_json(DIRS["final_quality"] / "final_quality_reconciliation_result.json", default={})
    final_editorial = read_json(DIRS["final_editorial"] / "final_editorial_polish_result.json", default={})
    failures = []
    if final_quality.get("approved") is not True:
        failures.append({"control": "final_quality_reconciliation", "result": final_quality})
    if final_editorial.get("approved") is not True:
        failures.append({"control": "final_editorial_polish", "result": final_editorial})
    if failures:
        write_json({"generated_at": now_stamp(), "failures": failures}, DIRS["pdf_handoff"] / "final_handoff_quality_control_failure.json")
        raise ValueError("Final handoff blocked because a strict final quality control did not approve.")


def assemble_final_markdown() -> Dict[str, Any]:
    assert_quality_controls_ready_for_handoff()
    sections = load_current_sections(prefer=("final_editorial", "final_quality", "approved_sections"))
    report_lines = ["# IFRS S1/S2 Sustainability-Related Financial Disclosures", ""]
    for section in SECTIONS:
        markdown = sections.get(section, "").strip()
        if not markdown:
            raise ValueError(f"Missing final section: {section}")
        report_lines.append(markdown)
        report_lines.append("")
    report = "\n".join(report_lines).strip() + "\n"
    assert_no_forbidden_language_in_report(report)
    report_path = DIRS["pdf_handoff"] / "approved_report_markdown.md"
    write_text(report, report_path)
    manifest = {
        "generated_at": now_stamp(),
        "report_path": str(report_path),
        "sections": {section: str(DIRS["final_editorial"] / f"final_{section_slug(section)}.md") for section in SECTIONS},
        "ready_for_pdf": True,
    }
    write_json(manifest, DIRS["pdf_handoff"] / "pdf_handoff_manifest.json")
    return manifest

pdf_handoff_manifest = assemble_final_markdown()
pdf_handoff_manifest


In [ ]:
# ============================================================
# CELL 18 — AUDIT SUMMARY
# ============================================================

def write_generation_audit_summary() -> Dict[str, Any]:
    section_results = read_json(OUTPUT_DIR / "section_generation_results.json", default=[])
    summary = {
        "generated_at": now_stamp(),
        "config": asdict(CONFIG),
        "input_summary": read_json(OUTPUT_DIR / "input_loading_summary.json", default={}),
        "section_generation_results": section_results,
        "quality_refinement": read_json(DIRS["quality_refinement"] / "quality_refinement_result.json", default={}),
        "final_quality": read_json(DIRS["final_quality"] / "final_quality_reconciliation_result.json", default={}),
        "final_editorial": read_json(DIRS["final_editorial"] / "final_editorial_polish_result.json", default={}),
        "connectivity": read_json(DIRS["connectivity"] / "whole_report_connectivity_judge.json", default={}),
        "pdf_handoff": read_json(DIRS["pdf_handoff"] / "pdf_handoff_manifest.json", default={}),
    }
    write_json(summary, OUTPUT_DIR / "generation_audit_summary.json")

    md_lines = ["# Generation Audit Summary", ""]
    md_lines.append(f"Generated at: {summary['generated_at']}")
    md_lines.append("")
    md_lines.append("## Section results")
    for result in section_results:
        md_lines.append(f"- {result.get('section_name')}: {result.get('status')} | score={result.get('section_score')} | approved={result.get('approved')}")
    md_lines.append("")
    md_lines.append(f"Final quality approved: {summary['final_quality'].get('approved')}")
    md_lines.append(f"Final editorial approved: {summary['final_editorial'].get('approved')}")
    md_lines.append(f"PDF ready: {summary['pdf_handoff'].get('ready_for_pdf')}")
    write_text("\n".join(md_lines), OUTPUT_DIR / "generation_audit_summary.md")
    return summary

generation_audit_summary = write_generation_audit_summary()
generation_audit_summary


In [ ]:
# ============================================================
# CELL 19 — NOTEBOOK SELF-CHECKS
# ============================================================

expected_files = [
    OUTPUT_DIR / "section_generation_results.json",
    DIRS["final_quality"] / "final_quality_reconciliation_result.json",
    DIRS["final_editorial"] / "final_editorial_polish_result.json",
    DIRS["pdf_handoff"] / "approved_report_markdown.md",
    OUTPUT_DIR / "generation_audit_summary.json",
]

self_check = {
    "generated_at": now_stamp(),
    "expected_files": {str(path): path.exists() for path in expected_files},
    "public_functions_intended_once": [
        "write_section_draft",
        "build_claims_register",
        "run_deterministic_gates",
        "run_llm_judges",
        "composite_approval_gate",
        "revise_section_minimally",
        "run_section_pipeline_langgraph",
        "final_qa_run_final_quality_reconciliation",
        "run_final_editorial_polish",
        "assemble_final_markdown",
    ],
}
write_json(self_check, OUTPUT_DIR / "notebook_self_check.json")
self_check
